In [2]:
!pip install tensorflow
!pip install keras


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 9.7 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
TrainingImagePath="Face Images/Final Training Images"
TestingImagePath="Face Images/Final Testing Images"

In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ImportError: DLL load failed while importing flags_pybind: An Application Control policy has blocked this file.

In [1]:


# Create a generator for TRAINING images
train_datagen = ImageDataGenerator(

    # Convert pixel values from 0-255 to 0-1
    # Example: 255 becomes 1.0, 128 becomes 0.502
    rescale=1./255,

    # Randomly flip images horizontally
    # Example: 👈 image becomes 👉 image
    horizontal_flip=True,

    # Move the image slightly up or down
    # 0.1 means up to 10% of the image height
    height_shift_range=0.1,

    # Move the image slightly left or right
    # 0.1 means up to 10% of the image width
    width_shift_range=0.1,

    # Randomly change image brightness
    # 0.5 = darker
    # 1.5 = brighter
    brightness_range=(0.5, 1.5),

    # Randomly zoom the image
    # [1, 1.5] means from normal size up to 1.5x zoom
    zoom_range=[1, 1.5],
)


# Create a generator for TESTING images
testing_datagen = ImageDataGenerator(

    # Only normalize pixel values from 0-255 to 0-1
    # We DON'T randomly modify testing images
    rescale=1./255
)

NameError: name 'ImageDataGenerator' is not defined

In [4]:
training_set = train_datagen.flow_from_directory(

    # Location of your training image folder
    directory=TrainingImagePath,

    # Resize every image to 64 × 64 pixels
    target_size=(64, 64),

    # Give 32 images to the CNN at a time
    batch_size=32,
)

Found 238 images belonging to 16 classes.


In [5]:
testing_set = testing_datagen.flow_from_directory( 
      # Location of your testing image folder
        directory=TestingImagePath,
    
        # Resize every image to 64 × 64 pixels
        target_size=(64, 64),
    
        # Give 32 images to the CNN at a time
        batch_size=32,
)

Found 64 images belonging to 16 classes.


In [6]:
# Asks Keras:

# "What number did you give to each class?"
testing_set.class_indices

{'face1': 0,
 'face10': 1,
 'face11': 2,
 'face12': 3,
 'face13': 4,
 'face14': 5,
 'face15': 6,
 'face16': 7,
 'face2': 8,
 'face3': 9,
 'face4': 10,
 'face5': 11,
 'face6': 12,
 'face7': 13,
 'face8': 14,
 'face9': 15}

In [7]:
# 👉 Takes that answer and saves it in a variable called trainClasses.
trainClasses=training_set.class_indices
trainClasses

{'face1': 0,
 'face10': 1,
 'face11': 2,
 'face12': 3,
 'face13': 4,
 'face14': 5,
 'face15': 6,
 'face16': 7,
 'face2': 8,
 'face3': 9,
 'face4': 10,
 'face5': 11,
 'face6': 12,
 'face7': 13,
 'face8': 14,
 'face9': 15}

In [8]:
# Create an empty dictionary
ResultMap = {}

# Go through each class name and its assigned number
# Example: 'face1' → 0, 'face10' → 1, etc.
for faceName, faceValue in trainClasses.items():

    # Store the class name and its number in ResultMap
    ResultMap[faceName] = faceValue

In [9]:
ResultMap

{'face1': 0,
 'face10': 1,
 'face11': 2,
 'face12': 3,
 'face13': 4,
 'face14': 5,
 'face15': 6,
 'face16': 7,
 'face2': 8,
 'face3': 9,
 'face4': 10,
 'face5': 11,
 'face6': 12,
 'face7': 13,
 'face8': 14,
 'face9': 15}

In [10]:
# Import required Keras layers
from keras.models import Sequential
from keras.layers import Convolution2D
from keras.layers import MaxPool2D
from keras.layers import Flatten
from keras.layers import Dense


# Create CNN model
classifier = Sequential()


# STEP 1: Convolution - extract features from images
classifier.add(
    Convolution2D(
        filters=10,
        kernel_size=(3, 3),
        strides=(1, 1),
        input_shape=(64, 64, 3),
        activation='relu'
    )
)


# STEP 2: Max Pooling - reduce feature map size
classifier.add(MaxPool2D(pool_size=(2, 2)))


# Second Convolution - detect more complex features
classifier.add(
    Convolution2D(
        filters=4,
        kernel_size=(5, 5),
        strides=(2, 2),
        activation='relu'
    )
)

# Second Max Pooling - reduce size again
classifier.add(MaxPool2D(pool_size=(2, 2)))


# STEP 3: Flatten - convert feature maps into 1D
classifier.add(Flatten())


# STEP 4: Dense layer - learn and classify features
classifier.add(Dense(64, activation='relu'))


# Output layer - 16 classes
classifier.add(Dense(16, activation='softmax'))


c:\Python314\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# Compile the CNN: tell the model how to learn and measure performance
classifier.compile(
    optimizer='adam',              # Helps the model improve its weights
    loss='binary_crossentropy',     # Measures how wrong the prediction is
    metrics=['accuracy']            # Shows how many predictions are correct
)

In [12]:
# Train the CNN model
classifier.fit(

    training_set,              # Training images

    steps_per_epoch=8,         # Use 8 batches of training images per epoch

    epochs=50,                 # Train the model 50 times over the data

    validation_data=testing_set,  # Use testing data to check performance

    validation_steps=2         # Use 2 batches for validation in each epoch
)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 433ms/step - accuracy: 0.0714 - loss: 0.6803 - val_accuracy: 0.0625 - val_loss: 0.5654
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 186ms/step - accuracy: 0.0840 - loss: 0.5021 - val_accuracy: 0.0625 - val_loss: 0.3794
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 207ms/step - accuracy: 0.0798 - loss: 0.3294 - val_accuracy: 0.0625 - val_loss: 0.2908
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 219ms/step - accuracy: 0.0840 - loss: 0.2749 - val_accuracy: 0.0781 - val_loss: 0.2719
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 281ms/step - accuracy: 0.0588 - loss: 0.2551 - val_accuracy: 0.0469 - val_loss: 0.2456
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 246ms/step - accuracy: 0.0462 - loss: 0.2510 - val_accuracy: 0.0781 - val_loss: 0.2438
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 199ms/step - accuracy: 0.0756 - loss: 0.2505 - val_accuracy: 0.0625 - val_loss: 0.2539
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 194ms/step - accuracy: 0.0840 - loss: 0.2494 - val_accuracy: 0.0625 - val_loss:

In [ ]:
# Import NumPy for working with numerical data
import numpy as np

# Import image functions from Keras
from keras.preprocessing import image


# Path of the image we want to test
testImage = r"Face Images\Final Testing Images\face8\3face8.jpg"


# Load the test image and resize it to 63x63 pixels
test_image = image.load_img(testImage, target_size=(63, 63))


# Display the loaded image
test_image